# EDG Organic Search CTR & Position Audit — analysis notebook

Read-only analysis of the Google Search Console exports captured July 13, 2026. The property is `sc-domain:edgpatioshade.com`; search type is Web.

## Context and methods

The notebook uses only standard-library Python and the CSV snapshots in the adjacent evidence folder. Query exports are incomplete because Search Console omits anonymized queries and may limit rows. Page rows are not additive because more than one URL can appear for one query.

In [1]:
import csv, json, re
from pathlib import Path

EVIDENCE = Path('docs/codex/organic-search-ctr-position-audit-2026-07-13-evidence')

def read_rows(path):
    with path.open(newline='') as handle:
        return list(csv.DictReader(handle))

def pct(value):
    return float(value.rstrip('%')) if value else None

## Baseline

In [2]:
baseline_rows = read_rows(EVIDENCE / 'last-3-months' / 'Chart.csv')
clicks = sum(int(row['Clicks']) for row in baseline_rows)
impressions = sum(int(row['Impressions']) for row in baseline_rows)
weighted_position = sum(float(row['Position'] or 0) * int(row['Impressions']) for row in baseline_rows) / impressions
baseline = {
    'clicks': clicks,
    'impressions': impressions,
    'ctr_percent': round(clicks / impressions * 100, 3),
    'weighted_position': round(weighted_position, 2),
}
print(json.dumps(baseline, indent=2))

{
  "clicks": 1042,
  "impressions": 95765,
  "ctr_percent": 1.088,
  "weighted_position": 17.26
}


## Query segmentation and suspected automated impressions

In [3]:
query_rows = read_rows(EVIDENCE / 'last-3-months-vs-previous' / 'Queries.csv')

def suspect(query):
    query = query.lower()
    return (
        query.startswith('automated retractable pergolas ')
        or query.startswith('deerfield il ')
        or query in {'magnatrack motorized retractable screens', 'patio enclosure supplier'}
    )

suspect_rows = [row for row in query_rows if suspect(row['Top queries'])]
suspect_impressions = sum(int(row['Last 3 months Impressions']) for row in suspect_rows)
suspect_summary = {
    'rows': len(suspect_rows),
    'clicks': sum(int(row['Last 3 months Clicks']) for row in suspect_rows),
    'current_impressions': suspect_impressions,
    'previous_impressions': sum(int(row['Previous 3 months Impressions']) for row in suspect_rows),
    'share_of_property_impressions_percent': round(suspect_impressions / impressions * 100, 2),
    'indicative_ctr_excluding_known_suspect_percent': round(clicks / (impressions - suspect_impressions) * 100, 2),
}
print(json.dumps(suspect_summary, indent=2))

{
  "rows": 262,
  "clicks": 0,
  "current_impressions": 16829,
  "previous_impressions": 2004,
  "share_of_property_impressions_percent": 17.57,
  "indicative_ctr_excluding_known_suspect_percent": 1.32
}


## Recent page movement

In [4]:
recent_pages = read_rows(EVIDENCE / 'recent-28-days-vs-previous' / 'Pages.csv')
movement = []
for row in recent_pages[:5]:
    url = row['Top pages']
    if url == 'https://edgpatioshade.com/':
        page = 'non-www-host-variant:/'
    else:
        page = url.replace('https://www.edgpatioshade.com', '') or '/'
    movement.append({
        'page': page,
        'click_delta': int(row['Last 28 days Clicks']) - int(row['Previous 28 days Clicks']),
        'impression_delta': int(row['Last 28 days Impressions']) - int(row['Previous 28 days Impressions']),
        'ctr_change_pp': round(pct(row['Last 28 days CTR']) - pct(row['Previous 28 days CTR']), 2),
        'position_change': round(float(row['Last 28 days Position']) - float(row['Previous 28 days Position']), 2),
    })
print('[')
for index, row in enumerate(movement):
    suffix = ',' if index < len(movement) - 1 else ''
    print('  ' + json.dumps(row) + suffix)
print(']')

[
  {"page": "/guides/magnatrack-screens-cost", "click_delta": -36, "impression_delta": 1126, "ctr_change_pp": -0.84, "position_change": 2.2},
  {"page": "/", "click_delta": -16, "impression_delta": 1131, "ctr_change_pp": -0.31, "position_change": 6.08},
  {"page": "non-www-host-variant:/", "click_delta": -12, "impression_delta": -451, "ctr_change_pp": -0.14, "position_change": -0.67},
  {"page": "/systems/shades", "click_delta": -19, "impression_delta": -647, "ctr_change_pp": -0.25, "position_change": 8.47},
  {"page": "/systems/enclosures", "click_delta": 4, "impression_delta": -190, "ctr_change_pp": 0.42, "position_change": 3.41}
]


## Takeaways

- The three-month headline improved strongly versus the previous period, but the latest 28-day period weakened: clicks fell while impressions rose, CTR fell, and average position worsened.
- Known templated zero-click query families materially distort property-wide impressions and position. The adjusted CTR is an indicative diagnostic, not a replacement Search Console KPI.
- The most actionable page losses are concentrated in the MagnaTrack cost guide, the two homepage host rows, and `/systems/shades`.
- Clean restaurant-patio-enclosure queries provide the best small first pilot because one correct route already owns the impressions.